# FVCOM Nest Forcing from Native NEMO Output

This tutorial demonstrates how to create FVCOM nest forcing files from native NEMO output using PyFVCOM2. The example uses AMM7 NEMO files split across `grid_T`, `grid_U`, `grid_V`, and `grid_W` outputs with curvilinear longitude and latitude coordinates.

## Overview

FVCOM nested grids require boundary forcing for elevation, temperature, salinity, and velocity. This tutorial shows how to:

1. Set up paths for the AMM7 NEMO and FVCOM sample data
2. Create the FVCOM grid and nest manager
3. Read native NEMO files with `NEMOReader`
4. Interpolate NEMO temperature, salinity, and velocity to the FVCOM nest
5. Generate a nest forcing file
6. Visualize the NEMO source grid and generated nest output

## Note on sea-surface height

The AMM7 NEMO sample directory used here does not include a sea-surface-height variable such as `sossheig`. The notebook therefore uses a zero-valued zeta interpolator so the FVCOM nest file can be written. For production forcing, replace that placeholder with a NEMO file containing sea-surface height and map `zeta` to the correct variable name.

## 1. Import Required Libraries

First, import the PyFVCOM2 readers, interpolators, nest tools, and plotting packages used in the workflow:

In [ ]:
from datetime import datetime, timedelta
import inspect
import os
from pathlib import Path
import sys

def find_pyfvcom2_checkout(start):
    for path in (start, *start.parents):
        if (path / 'pyfvcom2' / 'nemo_reader.py').exists():
            return path
    return None

pyfvcom2_checkout = find_pyfvcom2_checkout(Path.cwd().resolve())
if pyfvcom2_checkout is not None and str(pyfvcom2_checkout) not in sys.path:
    sys.path.insert(0, str(pyfvcom2_checkout))

import numpy as np
import xarray as xr
from netCDF4 import Dataset

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.interpolate import LinearNDInterpolator

import pyfvcom2
from pyfvcom2.date_utils import create_datetime_array
from pyfvcom2.exceptions import PyFVCOM2ValueError
from pyfvcom2.grid import create_grid
from pyfvcom2.interpolation import NEMOInterpolator
from pyfvcom2.interpolation_coordinates import InterpolationCoordinates
from pyfvcom2.nemo_reader import NEMOReader
from pyfvcom2.nest import NestManager
from pyfvcom2.plotting import FVCOMPlotter

print(f'Using pyfvcom2 from {Path(pyfvcom2.__file__).resolve()}')
print(f'NEMOReader signature: {inspect.signature(NEMOReader.__init__)}')

## 2. Configuration and Setup

Define the paths for the AMM7 NEMO files and the FVCOM Tamar Estuary grid used by the cookbook examples:

In [ ]:
data_dir = Path(os.path.expanduser('~/data/pyfvcom2_doc'))

# Native NEMO output split by grid.
nemo_data_dir = data_dir / 'AMM7_NEMO'
nemo_files = {
    'T': nemo_data_dir / 'amm7_1d_19820101_19820131_grid_T.nc',
    'U': nemo_data_dir / 'amm7_1d_19820101_19820131_grid_U.nc',
    'V': nemo_data_dir / 'amm7_1d_19820101_19820131_grid_V.nc',
    'W': nemo_data_dir / 'amm7_1d_19820101_19820131_grid_W.nc',
}

# Optional mesh mask. The sample directory does not include one, but the
# reader will use it if you add mesh_mask.nc beside the NEMO files.
mesh_mask_file = nemo_data_dir / 'mesh_mask.nc'
mask_file_paths = mesh_mask_file if mesh_mask_file.exists() else None

# FVCOM grid configuration.
fvcom_data_dir = data_dir / 'FVCOM_tamar_estuary'
grid_file = fvcom_data_dir / 'tamar_v2_grd.dat'
obc_file = fvcom_data_dir / 'tamar_v2_obc.dat'
sigma_file = fvcom_data_dir / 'sigma_gen.dat'

print(f'NEMO data directory: {nemo_data_dir}')
for grid_name, file_path in nemo_files.items():
    print(f'  {grid_name}: {file_path.name} ({"found" if file_path.exists() else "missing"})')
print(f'Mesh mask: {mask_file_paths if mask_file_paths else "not supplied"}')
print(f'Grid file: {grid_file}')
print(f'Open boundary file: {obc_file}')
print(f'Sigma file: {sigma_file}')

## 3. Time Period Definition

The AMM7 sample files contain daily data at noon from 1 January 1982 to 31 January 1982. To keep the tutorial light, use three daily output times:

In [ ]:
start_date_time = datetime(1982, 1, 1, 12)
end_date_time = datetime(1982, 1, 3, 12)

date_times = create_datetime_array(
    start_date_time,
    end_date_time,
    timedelta(days=1),
)

print(f'Start time: {start_date_time}')
print(f'End time: {end_date_time}')
print(f'Number of time steps: {len(date_times)}')
print('Time step interval: 1 day')

## 4. FVCOM Grid Creation

Load the FVCOM mesh, sigma levels, and open-boundary definition:

In [ ]:
fvcom_grid = create_grid(
    grid_file,
    mesh_type='fvcom',
    sigma_file=sigma_file,
    coordinate_system='cartesian',
    epsg_code='32630',
    obc_filename=obc_file,
)

print('Grid loaded successfully!')
print(f'Number of nodes: {fvcom_grid.n_nodes}')
print(f'Number of elements: {fvcom_grid.n_elements}')
print(f'Number of sigma levels: {fvcom_grid.n_sigma_levels}')
print(f'Number of open boundaries: {len(fvcom_grid.open_boundaries)}')
print(f'Longitude range: [{fvcom_grid.lon_nodes.min():.4f}, {fvcom_grid.lon_nodes.max():.4f}]')
print(f'Latitude range: [{fvcom_grid.lat_nodes.min():.4f}, {fvcom_grid.lat_nodes.max():.4f}]')

## 5. Nest Manager Setup

Create the `NestManager`, which collects the open-boundary nodes and adjoining grid bands, stores interpolated forcing arrays, and writes the final FVCOM nest file:

In [ ]:
nest_manager = NestManager(
    fvcom_grid,
    num_grid_bands=2,
    weights_calculation_method='linear',
)

nest_manager.set_dates(date_times)

print('NestManager created successfully!')
print('Number of grid bands: 2')
print('Interpolation method: linear')
print(f'Nest nodes: {len(nest_manager.get_all_nest_nodes())}')
print(f'Nest elements: {len(nest_manager.get_all_nest_elements())}')

## 6. Inspect the Native NEMO Files

`NEMOReader` keeps the T, U, V, and W grids separate. It reads the curvilinear coordinates from each variable's metadata, applies masks when a mesh-mask file is supplied, and reconstructs T-cell vertical coordinates from `e3t` where available:

In [ ]:
nemo_reader_kwargs = {}
nemo_reader_parameters = inspect.signature(NEMOReader.__init__).parameters
if 'mask_file_paths' in nemo_reader_parameters:
    nemo_reader_kwargs['mask_file_paths'] = mask_file_paths
elif 'mask_file_path' in nemo_reader_parameters:
    nemo_reader_kwargs['mask_file_path'] = mask_file_paths
elif mask_file_paths is not None:
    raise TypeError(
        'The loaded NEMOReader does not accept a mask file path. '
        'Restart the kernel and make sure it imports this checkout.'
    )

nemo_reader = NEMOReader(nemo_files, **nemo_reader_kwargs)

print(f'NEMO grids loaded: {nemo_reader.grid_names}')
print(f'Zero-value mask variables: {sorted(nemo_reader.zero_value_mask_var_names)}')
for grid_name in nemo_reader.grid_names:
    span = nemo_reader.time_span(grid_name)
    print(f'{grid_name}: {span["count"]} times from {span["start"]} to {span["end"]}')

for var_name in ['votemper', 'vosaline', 'uo', 'vo']:
    grid_name = nemo_reader.grid_for_variable(var_name)
    lons = nemo_reader.lons_for_variable(var_name, grid_name)
    lats = nemo_reader.lats_for_variable(var_name, grid_name)
    depths = nemo_reader.depth_levels_for_variable(var_name, grid_name)
    print(
        f'{var_name}: grid={grid_name}, lon/lat shape={lons.shape}, '
        f'depth range={depths[0]:.2f}-{depths[-1]:.2f} m'
    )

## 7. Create the NEMO Interpolator

The default mapping uses the native NEMO variable names in the AMM7 sample files:

- `temp` -> `votemper`
- `salinity` -> `vosaline`
- `u` -> `uo`
- `v` -> `vo`
- `zeta` -> `sossheig`

This sample does not include `sossheig`, so the zeta step below uses a documented placeholder.

In [ ]:
nemo_interpolator = NEMOInterpolator(nemo_reader)

print('Default FVCOM-to-NEMO mapping:')
for fvcom_var, nemo_var in nemo_interpolator.fvcom_to_nemo_var_names.items():
    try:
        grid_name = nemo_reader.grid_for_variable(nemo_var)
        status = f'available on {grid_name}'
    except PyFVCOM2ValueError:
        status = 'not found in this sample'
    print(f'  {fvcom_var:8s} -> {nemo_var:10s} {status}')

## 8. Add Sea-Surface Height Forcing

`NestManager.create_forcing_file()` writes a complete FVCOM nest file and expects `zeta` to be present. If your NEMO files include sea-surface height, call `nest_manager.add_forcing_data(nemo_interpolator, 'zeta', 'node')` after mapping `zeta` to the correct variable name.

For this AMM7 sample, use a zero-valued zeta placeholder so the rest of the native NEMO workflow can be demonstrated:

In [ ]:
class ConstantZetaInterpolator:
    """Small notebook helper for samples without sea-surface height."""

    def __init__(self, value=0.0):
        self.value = np.float32(value)

    def interpolate(
        self,
        coordinates: InterpolationCoordinates,
        fvcom_var_name: str,
    ) -> np.ndarray:
        if fvcom_var_name != 'zeta':
            raise ValueError('ConstantZetaInterpolator only supports zeta')
        dates = np.atleast_1d(coordinates.dates)
        return np.full((len(dates), len(coordinates.x1)), self.value, dtype=np.float32)

try:
    nemo_reader.grid_for_variable(nemo_interpolator.fvcom_to_nemo_var_names['zeta'])
    nest_manager.add_forcing_data(nemo_interpolator, 'zeta', horizontal_position='node')
    print('Added zeta from NEMO sea-surface-height data')
except PyFVCOM2ValueError:
    zero_zeta = ConstantZetaInterpolator(0.0)
    nest_manager.add_forcing_data(zero_zeta, 'zeta', horizontal_position='node')
    print('No NEMO sea-surface-height variable found; added zero zeta placeholder')

## 9. Add 3D NEMO Forcing Variables

Add NEMO velocities at FVCOM element centres and temperature/salinity at FVCOM nodes. NEMO `uo` and `vo` are interpolated from their native U and V grids and rotated onto east/north components before being returned as FVCOM `u` and `v`:

In [ ]:
forcing_variables = [
    ('u', 'element'),
    ('v', 'element'),
    ('temp', 'node'),
    ('salinity', 'node'),
]

print('Adding 3D forcing variables:')
for fvcom_var, position in forcing_variables:
    nest_manager.add_forcing_data(
        nemo_interpolator,
        fvcom_var,
        horizontal_position=position,
    )
    data = nest_manager.get_forcing_data(fvcom_var)
    print(f'  {fvcom_var:8s} at {position:7s}: shape={data.shape}')

print('3D NEMO forcing data added successfully')

## 10. Generate the FVCOM Nest Forcing File

Write the final FVCOM nest forcing file. `NestManager` automatically derives depth-averaged `ua` and `va` from the interpolated 3D velocities:

In [ ]:
file_name = 'tamar_v2_nest_forcing_from_nemo.nc'

nest_manager.create_forcing_file(file_name, nest_type=3)

print(f'Nest forcing file created: {file_name}')
print(f'File size: {os.path.getsize(file_name) / (1024 * 1024):.1f} MB')

## 11. Visualize the NEMO Source Grid

Plot the first surface temperature field from the native NEMO T grid and overlay the FVCOM nest nodes. This uses `NEMOReader.get_var()` so the same inferred land mask is used here and during interpolation. For display only, the native NEMO values are interpolated onto a regular plotting grid and shown with `pcolormesh`; this avoids drawing artefacts from the coarse native structured columns while keeping the forcing interpolation on the native NEMO grid. The colour scale can be controlled with `temp_vmin` and `temp_vmax`:

In [ ]:
source_temp = nemo_reader.get_var(
    'votemper',
    target_datetime=date_times[0],
    grid='T',
    depth_index=0,
)
source_lon = nemo_reader.lons_for_variable('votemper', 'T')
source_lat = nemo_reader.lats_for_variable('votemper', 'T')

def one_dimensional_cell_bounds(values):
    values = np.asarray(values, dtype=float)
    bounds = np.empty(values.size + 1, dtype=float)
    bounds[1:-1] = 0.5 * (values[:-1] + values[1:])
    bounds[0] = values[0] - 0.5 * (values[1] - values[0])
    bounds[-1] = values[-1] + 0.5 * (values[-1] - values[-2])
    return bounds

plot_extent = [
    fvcom_grid.lon_nodes.min() - 0.3,
    fvcom_grid.lon_nodes.max() + 0.3,
    fvcom_grid.lat_nodes.min() - 0.3,
    fvcom_grid.lat_nodes.max() + 0.3,
]
visible_source = (
    (source_lon >= plot_extent[0])
    & (source_lon <= plot_extent[1])
    & (source_lat >= plot_extent[2])
    & (source_lat <= plot_extent[3])
)
visible_source_temp = np.where(visible_source, source_temp, np.nan)

source_temp_min = float(np.nanmin(visible_source_temp))
source_temp_max = float(np.nanmax(visible_source_temp))

# Set these to fixed values if you want a consistent colour scale.
temp_vmin = source_temp_min
temp_vmax = source_temp_max
temp_cmap = 'viridis'

print(f'Visible NEMO surface temperature range: {source_temp_min:.3f} to {source_temp_max:.3f} degC')
print(f'Plot colour scale: {temp_vmin:.3f} to {temp_vmax:.3f} degC')

display_nx = 500
display_ny = 500
display_lon = np.linspace(plot_extent[0], plot_extent[1], display_nx)
display_lat = np.linspace(plot_extent[2], plot_extent[3], display_ny)
display_lon_grid, display_lat_grid = np.meshgrid(display_lon, display_lat)

source_points = np.column_stack((source_lon[visible_source & np.isfinite(source_temp)], source_lat[visible_source & np.isfinite(source_temp)]))
source_values = source_temp[visible_source & np.isfinite(source_temp)]
display_points = np.column_stack((display_lon_grid.ravel(), display_lat_grid.ravel()))

display_temp = LinearNDInterpolator(source_points, source_values)(display_points)
display_temp = display_temp.reshape(display_lon_grid.shape)

display_lon_bounds = one_dimensional_cell_bounds(display_lon)
display_lat_bounds = one_dimensional_cell_bounds(display_lat)
source_temp_plot = np.ma.masked_invalid(display_temp)
nest_nodes = nest_manager.get_all_nest_nodes()

fig = plt.figure(figsize=(10, 8))
ax = plt.axes(projection=ccrs.PlateCarree())
pc = ax.pcolormesh(
    display_lon_bounds,
    display_lat_bounds,
    source_temp_plot,
    shading='flat',
    transform=ccrs.PlateCarree(),
    cmap=temp_cmap,
    vmin=temp_vmin,
    vmax=temp_vmax,
)
ax.scatter(
    fvcom_grid.lon_nodes[nest_nodes],
    fvcom_grid.lat_nodes[nest_nodes],
    s=6,
    c='red',
    transform=ccrs.PlateCarree(),
    label='FVCOM nest nodes',
    zorder=5,
)
ax.add_feature(cfeature.LAND.with_scale('10m'), facecolor='white', edgecolor='black', zorder=3)
ax.coastlines(resolution='10m', zorder=4)
ax.set_extent(plot_extent)
ax.legend(loc='upper right')
plt.colorbar(pc, ax=ax, label='NEMO surface temperature (degC)')
plt.title('AMM7 NEMO source grid and FVCOM nest nodes')
plt.show()

## 12. Visualize Generated Nest Forcing

Open the generated FVCOM nest file and plot the interpolated surface temperature and current speed on the nest points. The temperature panel reuses the source-plot colour scale so the two temperature figures can be compared directly:

In [ ]:
nest_ds = Dataset(file_name)
fvcom_plotter = FVCOMPlotter(fvcom_grid)

surface_temp = nest_ds.variables['temp'][0, 0, :]
u_surface = nest_ds.variables['u'][0, 0, :]
v_surface = nest_ds.variables['v'][0, 0, :]
speed_surface = np.hypot(u_surface, v_surface)

nested_temp_min = float(np.nanmin(surface_temp))
nested_temp_max = float(np.nanmax(surface_temp))
print(f'Nested surface temperature range: {nested_temp_min:.3f} to {nested_temp_max:.3f} degC')
print(f'Plot colour scale: {temp_vmin:.3f} to {temp_vmax:.3f} degC')

nest_nodes = nest_manager.get_all_nest_nodes()
nest_elements = nest_manager.get_all_nest_elements()

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 6),
    subplot_kw={'projection': ccrs.PlateCarree()},
)

sc0 = axes[0].scatter(
    fvcom_grid.lon_nodes[nest_nodes],
    fvcom_grid.lat_nodes[nest_nodes],
    c=surface_temp,
    s=10,
    cmap=temp_cmap,
    vmin=temp_vmin,
    vmax=temp_vmax,
    transform=ccrs.PlateCarree(),
)
axes[0].coastlines(resolution='10m')
axes[0].set_title('Interpolated surface temperature')
plt.colorbar(sc0, ax=axes[0], label='degC')

sc1 = axes[1].scatter(
    fvcom_grid.lon_elements[nest_elements],
    fvcom_grid.lat_elements[nest_elements],
    c=speed_surface,
    s=10,
    cmap='magma',
    transform=ccrs.PlateCarree(),
)
axes[1].coastlines(resolution='10m')
axes[1].set_title('Interpolated surface current speed')
plt.colorbar(sc1, ax=axes[1], label='m/s')

for ax in axes:
    ax.set_extent([
        fvcom_grid.lon_nodes.min() - 0.05,
        fvcom_grid.lon_nodes.max() + 0.05,
        fvcom_grid.lat_nodes.min() - 0.05,
        fvcom_grid.lat_nodes.max() + 0.05,
    ])

plt.tight_layout()
plt.show()

## 13. Inspect the Output File

Finally, inspect the generated FVCOM nest file dimensions and variables:

In [ ]:
print('Generated forcing file structure:')
print('\nDimensions:')
for dim_name, dim in nest_ds.dimensions.items():
    size = len(dim) if not dim.isunlimited() else 'unlimited'
    print(f'  {dim_name}: {size}')

print('\nKey forcing variables:')
for var_name in ['zeta', 'ua', 'va', 'u', 'v', 'temp', 'salinity']:
    var = nest_ds.variables[var_name]
    print(f'  {var_name:8s}: shape={var.shape}, dimensions={var.dimensions}')

nest_ds.close()

## Summary

This notebook demonstrated the native NEMO forcing workflow:

- `NEMOReader` reads AMM7 `grid_T`, `grid_U`, `grid_V`, and `grid_W` files without converting them to a regular lon/lat grid.
- `NEMOInterpolator` interpolates curvilinear NEMO fields horizontally, vertically, and in time onto FVCOM nest coordinates.
- Native NEMO U/V velocities are rotated onto FVCOM east/north components.
- `NestManager` writes the FVCOM nest forcing file and derives depth-averaged `ua` and `va`.

For production use, supply a NEMO sea-surface-height file and, where available, a NEMO mesh-mask file so land and bottom cells are masked explicitly.